In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

In [ ]:
# Make sure file is inside Google Drive
file_path = '/content/drive/MyDrive/student-mat.csv'

stud = pd.read_csv(file_path)

print("Total Students:", len(stud))
stud.head()

In [ ]:
pd.set_option('display.max_columns', None)

stud.info()
stud.describe()
stud.isnull().sum()

In [ ]:
# Missing values
sns.heatmap(stud.isnull(), cmap="viridis", yticklabels=False)
plt.show()

# Gender
sns.countplot(x='sex', data=stud)
plt.show()

# Age distribution
sns.kdeplot(data=stud, x='age', fill=True)
plt.show()

# Age vs Gender
sns.countplot(x='age', hue='sex', data=stud)
plt.show()

# Address
sns.countplot(x='address', data=stud)
plt.show()

# Age vs Grade
sns.boxplot(x='age', y='G3', data=stud)
plt.show()

# Urban vs Rural Grades
sns.kdeplot(stud[stud['address']=='U']['G3'], label='Urban', fill=True)
sns.kdeplot(stud[stud['address']=='R']['G3'], label='Rural', fill=True)
plt.legend()
plt.show()

In [ ]:
numeric = stud.select_dtypes(include=np.number)
print(numeric.corr()['G3'].sort_values())

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in stud.select_dtypes(include='object').columns:
    stud[col] = le.fit_transform(stud[col])

In [ ]:
stud = stud.drop(['school', 'G1', 'G2'], axis=1, errors='ignore')

top_features = stud.corr().abs()['G3'].sort_values(ascending=False)[:9]
stud = stud[top_features.index]

print(stud.head())

In [ ]:
from sklearn.model_selection import train_test_split

X = stud.drop('G3', axis=1)
y = stud['G3']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
baseline = np.median(y_train)
baseline_preds = [baseline] * len(y_test)

mae = np.mean(abs(baseline_preds - y_test))
rmse = np.sqrt(np.mean((baseline_preds - y_test)**2))

print("Baseline MAE:", mae)
print("Baseline RMSE:", rmse)

In [ ]:
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

def evaluate_models():
    models = {
        "Linear": LinearRegression(),
        "Elastic": ElasticNet(),
        "RandomForest": RandomForestRegressor(),
        "ExtraTrees": ExtraTreesRegressor(),
        "SVM": SVR(),
        "GradientBoost": GradientBoostingRegressor()
    }

    results = []

    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        mae = np.mean(abs(pred - y_test))
        rmse = np.sqrt(np.mean((pred - y_test)**2))

        results.append([name, mae, rmse])

    return pd.DataFrame(results, columns=['Model', 'MAE', 'RMSE'])

results = evaluate_models()
print(results)

In [ ]:
results.set_index('Model')[['MAE','RMSE']].plot(kind='bar')
plt.title("Model Comparison")
plt.show()

In [ ]:
model = RandomForestRegressor()
model.fit(X_train, y_train)

pred = model.predict(X_test)

plt.scatter(y_test, pred)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted")
plt.show()